# CS549 Machine Learning
# Assignment 8: Optimization of Deep Neural Networks

**Total points: 15**

In this assignment, you will implement a multiple layer feed-forward neural network for a multi-class classification task.

In [3]:
import torch
torch.manual_seed(0)
torch.use_deterministic_algorithms(True)

from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision.transforms import ToTensor

## Task 1: Build a deep neural network model
**Points: 3**

Implement the `NeuralNetModel1` class. The model takes a $28\times 28$ grey-scale image as input, and pass it through a deep neural network.

The network has 2 hidden layers and 1 output layers, whose sizes are: 512 -> 512 -> 10. That is, the number of output classes is 10. The activation function for each hidden layer is `ReLU`.

The input image is first passed through a `nn.Flatten()` layer so that a 2D tensor becomes 1D.

In [16]:
class NeuralNetModel1(nn.Module):
    def __init__(self):
        super(NeuralNetModel1, self).__init__()
        ### START YOUR CODE ###
        self.flatten = nn.Flatten() # Use nn.Flatten()
        self.linear_relu_stack = nn.Sequential(
            nn.Linear(28 * 28, 512), # Input size is 28*28
            nn.ReLU(), # ReLU
            nn.Linear(512, 512), # 512 -> 512
            nn.ReLU(), # ReLU
            nn.Linear(512, 10), # 512 -> 10
        )
        ### END YOUR CODE ###

    def forward(self, x):
        ### START YOUR CODE ###
        x = self.flatten(x) # Call self.flatten()
        logits = self.linear_relu_stack(x) # Call self.linear_relu_stack()
        ### END YOUR CODE ###

        return logits

In [17]:
# Do not change the test code here
sample_input = torch.randn(5, 28, 28)
print('input size:', sample_input.size())

model1 = NeuralNetModel1()
with torch.no_grad():
    output = model1(sample_input)
print('output size:', output.size())

input size: torch.Size([5, 28, 28])
output size: torch.Size([5, 10])


**Expected output**:

input size: torch.Size([5, 28, 28])\
output size: torch.Size([5, 10])

---

## Task 2: Use dataloader
**Points: 1**

Download the FashionMNIST dataset provided by PyTorch to the folder "data", which takes some time for the first time execution.
Use the `DataLoader` module to wrap the loaded training and test data. Specify the `batch_size` correctly for both training and test dataloader.

See <https://pytorch.org/docs/stable/data.html#torch.utils.data.DataLoader> for more information.

In [18]:
training_data = datasets.FashionMNIST(
    root="data",
    train=True, # True
    download=True,
    transform=ToTensor()
)

test_data = datasets.FashionMNIST(
    root="data",
    train=False, # False
    download=True,
    transform=ToTensor()
)

batch_size = 64

### START YOUR CODE ###
train_loader = DataLoader(training_data, batch_size= batch_size) # Specify data source and batch size correctly
test_loader = DataLoader(test_data, batch_size=batch_size)
### END YOUR CODE ###

100%|████████████████████████████████████████████████████| 26421880/26421880 [00:58<00:00, 449872.35it/s]


Extracting data/FashionMNIST/raw/train-images-idx3-ubyte.gz to data/FashionMNIST/raw



100%|██████████████████████████████████████████████████████████| 29515/29515 [00:00<00:00, 179285.29it/s]


Extracting data/FashionMNIST/raw/train-labels-idx1-ubyte.gz to data/FashionMNIST/raw



100%|██████████████████████████████████████████████████████| 4422102/4422102 [00:11<00:00, 394435.29it/s]


Extracting data/FashionMNIST/raw/t10k-images-idx3-ubyte.gz to data/FashionMNIST/raw



100%|███████████████████████████████████████████████████████████| 5148/5148 [00:00<00:00, 7994178.82it/s]

Extracting data/FashionMNIST/raw/t10k-labels-idx1-ubyte.gz to data/FashionMNIST/raw



In [30]:
# Do not change the test code here
print('Training data size:', len(training_data))
print('Testing data size:', len(test_data))

count = 0
for batch in train_loader:
    X, y = batch
    print('X size:', X.size())
    print('y size:', y.size())
    count += 1
    if count > 0:
        break

Training data size: 60000
Testing data size: 10000
X size: torch.Size([64, 1, 28, 28])
y size: torch.Size([64])


**Expected output**:

Training data size: 60000\
Testing data size: 10000\
X size: torch.Size([64, 1, 28, 28])\
y size: torch.Size([64])

## Task 3: Define loss and optimizer
**Points: 1**

Use `nn.CrossEntropyLoss()` as the loss function, and use `torch.optim.SGD()` as the optimizer. Specify the arguments for `SGD()`, including the learning rate correctly.

See <https://pytorch.org/docs/stable/generated/torch.nn.CrossEntropyLoss.html> and <https://pytorch.org/docs/stable/optim.html> for more information.

In [31]:
learning_rate = 1e-3

### START YOUR CODE ###
loss_fn = nn.CrossEntropyLoss()
optimizer_sgd = torch.optim.SGD(model1.parameters(), lr = learning_rate )
### END YOUR CODE ###

In [32]:
# Do not change the test code here
print(loss_fn)
print(type(optimizer_sgd))

CrossEntropyLoss()
<class 'torch.optim.sgd.SGD'>


**Expected output**:

CrossEntropyLoss()
<class 'torch.optim.sgd.SGD'>

---

## Task 4: Implement train and test functions
**Points: 6**

Implement the code for training the model in `train()`. Implement the code for testing the model in `test()`. For the backpropagation step, you need to first zero out all gradients by calling `optimizer.zero_grad()` before carrying out `backward()` and `step()` to update parameters.

In `test()`, you need to calculate the number of correct prediction in the current batch, and add it to the `correct` variable.
Finally, you need to divide `correct` by the total number of test examples to obtain the test accuracy.

In [36]:
def train_loop(dataloader, model, loss_fn, optimizer, verbose=True):
    for i, (X, y) in enumerate(dataloader):
        # Compute prediction and loss
        ### START YOUR CODE ###
        pred = model(X)# Get the prediction output from model
        loss = loss_fn(pred,y) # compute loss by calling loss_fn()
        ### END YOUR CODE ###

        # Backpropagation
        ### START YOUR CODE ###
        optimizer.zero_grad() # zero_grad()
        loss.backward() # backward()
        optimizer.step() # step()
        ### END YOUR CODE ###

        if verbose and i % 100 == 0:
            loss = loss.item()
            current_step = i * len(X)
            print(f"loss: {loss:>7f}  [{current_step:>5d}/{len(dataloader.dataset):>5d}]")

In [37]:
@torch.no_grad()
def test_loop(dataloader, model, loss_fn):
    test_loss, correct = 0, 0

    for X, y in dataloader:
        ### START YOUR CODE ###
        pred = model(X) # Similar to how it is computed in train()
        loss = loss_fn(pred,y)
        test_loss += loss.item()
        correct += (pred.argmax(1) == y).type(torch.float).sum().item() # Add the number of correct prediction in the current batch to `correct`
        ### END YOUR CODE ###

    test_loss /= len(dataloader)
    ### START YOUR CODE ###
    test_acc = correct / len(dataloader.dataset) # Use `correct` to compute accuracy
    ### END YOUR CODE ###

    print(f"Test Error: \n Accuracy: {(100*test_acc):>0.1f}%, Avg loss: {test_loss:>8f} \n")

Next, execute the following cell to start the training and testing loop. Make sure that the cell containing the loss function and optimizers has already been executed.

In [38]:
model1 = NeuralNetModel1() # Reset the model
### START YOUR CODE ###
optimizer_sgd = torch.optim.SGD(model1.parameters(), lr=learning_rate)  # Because the model1 is reset, the optimizer also needs redefined.
### END YOUR CODE ###

epochs = 10
for t in range(epochs):
    print(f"Epoch {t+1}\n-------------------------------")
    ### START YOUR CODE ###
    train_loop(train_loader, model1, loss_fn, optimizer_sgd, verbose=True) # Use verbose=False, if you want to see less information
    test_loop(test_loader, model1, loss_fn)
    ### END YOUR CODE ###

print("Done!")

Epoch 1
-------------------------------
loss: 2.297476  [    0/60000]
loss: 2.285110  [ 6400/60000]
loss: 2.266848  [12800/60000]
loss: 2.267637  [19200/60000]
loss: 2.236075  [25600/60000]
loss: 2.220525  [32000/60000]
loss: 2.222497  [38400/60000]
loss: 2.186472  [44800/60000]
loss: 2.195663  [51200/60000]
loss: 2.162702  [57600/60000]
Test Error: 
 Accuracy: 51.1%, Avg loss: 2.144640 

Epoch 2
-------------------------------
loss: 2.156979  [    0/60000]
loss: 2.138639  [ 6400/60000]
loss: 2.081138  [12800/60000]
loss: 2.103253  [19200/60000]
loss: 2.031258  [25600/60000]
loss: 1.986616  [32000/60000]
loss: 2.012911  [38400/60000]
loss: 1.927092  [44800/60000]
loss: 1.952730  [51200/60000]
loss: 1.868983  [57600/60000]
Test Error: 
 Accuracy: 56.0%, Avg loss: 1.858549 

Epoch 3
-------------------------------
loss: 1.897680  [    0/60000]
loss: 1.856898  [ 6400/60000]
loss: 1.742826  [12800/60000]
loss: 1.784701  [19200/60000]
loss: 1.664544  [25600/60000]
loss: 1.631789  [32000/600

**Expected output**

The test accuracy from the last epoch should be above 70%.

---

Next, train an ADAM optimizer. Note that the model needs be reset.

In [39]:
model1 = NeuralNetModel1() # Reset the model

### START YOUR CODE ###
optimizer_adam = torch.optim.Adam(model1.parameters(), lr = learning_rate )
### END YOUR CODE ###

epochs = 10
for t in range(epochs):
    print(f"Epoch {t+1}\n-------------------------------")
    ### START YOUR CODE ###
    train_loop(train_loader, model1, loss_fn, optimizer_sgd, verbose=True) # Use verbose=False, if you want to see less information
    test_loop(test_loader, model1, loss_fn)
    ### END YOUR CODE ###

print("Done!")

Epoch 1
-------------------------------
loss: 2.300810  [    0/60000]
loss: 2.308928  [ 6400/60000]
loss: 2.295600  [12800/60000]
loss: 2.297518  [19200/60000]
loss: 2.305246  [25600/60000]
loss: 2.292712  [32000/60000]
loss: 2.304222  [38400/60000]
loss: 2.300854  [44800/60000]
loss: 2.297331  [51200/60000]
loss: 2.298435  [57600/60000]
Test Error: 
 Accuracy: 7.9%, Avg loss: 2.300105 

Epoch 2
-------------------------------
loss: 2.300810  [    0/60000]
loss: 2.308928  [ 6400/60000]
loss: 2.295600  [12800/60000]
loss: 2.297518  [19200/60000]
loss: 2.305246  [25600/60000]
loss: 2.292712  [32000/60000]
loss: 2.304222  [38400/60000]
loss: 2.300854  [44800/60000]
loss: 2.297331  [51200/60000]
loss: 2.298435  [57600/60000]
Test Error: 
 Accuracy: 7.9%, Avg loss: 2.300105 

Epoch 3
-------------------------------
loss: 2.300810  [    0/60000]
loss: 2.308928  [ 6400/60000]
loss: 2.295600  [12800/60000]
loss: 2.297518  [19200/60000]
loss: 2.305246  [25600/60000]
loss: 2.292712  [32000/60000

**Expected output**:

You can find that the training converges much faster using ADAM.

---

## Task 5: Add batchnorm and dropout
**Points: 4**

Use `torch.nn.BatchNorm1d()` and `nn.Dropout()` after the ReLU activation of each hidden layer. `Batchnorm1d()` takes the size of previous activation as input. `Dropout()` takes the probability of dropout as input.

For more information, see <https://pytorch.org/docs/stable/generated/torch.nn.BatchNorm1d.html> and <https://pytorch.org/docs/stable/generated/torch.nn.Dropout.html>.

In [40]:
class NeuralNetModel2(nn.Module):
    def __init__(self, dropout = 0.1): # Note the additional dropout parameter here
        """
        :param dropout: float, the probability of dropout
        """
        super(NeuralNetModel2, self).__init__()
        ### START YOUR CODE ###
        self.flatten = nn.Flatten()
        self.linear_relu_stack = nn.Sequential(
            nn.Linear(28*28, 512),  # Input size is 28*28
            nn.ReLU(),              # ReLU
            nn.BatchNorm1d(512),    # Batch normalization
            nn.Dropout(dropout),    # Dropout, use the `dropout` parameter

            nn.Linear(512, 256),    # 512 -> 256
            nn.ReLU(),              # ReLU
            nn.BatchNorm1d(256),    # Batch normalization
            nn.Dropout(dropout),    # Dropout with the given probability

            nn.Linear(256, 10)      # 256 -> 10
        )
        ### END YOUR CODE ###

    def forward(self, x):
        ### START YOUR CODE ###
        x = self.flatten(x)
        logits = self.linear_relu_stack(x)
        ### END YOUR CODE ###

        return logits

In the following cell, test with different `dropout` rates, and observe how that affects the test accuracy.

In [41]:
### START YOUR CODE ###
model2 = NeuralNetModel2() # Call NeuralNetModel2() with the dropout value you want to try
optimizer = torch.optim.SGD(model2.parameters(), lr=learning_rate) # You may try Adam/SGD optimizer
### END YOUR CODE ###

epochs = 10
for t in range(epochs):
    print(f"Epoch {t+1}\n-------------------------------")
    ### START YOUR CODE ###
    train_loop(train_loader, model2, loss_fn, optimizer_sgd, verbose=True) # Use verbose=False, if you want to see less information
    test_loop(test_loader, model2, loss_fn)
    ### END YOUR CODE ###

print("Done!")

Epoch 1
-------------------------------
loss: 2.439142  [    0/60000]
loss: 2.402408  [ 6400/60000]
loss: 2.455243  [12800/60000]
loss: 2.567568  [19200/60000]
loss: 2.456326  [25600/60000]
loss: 2.436426  [32000/60000]
loss: 2.421608  [38400/60000]
loss: 2.467247  [44800/60000]
loss: 2.479336  [51200/60000]
loss: 2.462527  [57600/60000]
Test Error: 
 Accuracy: 8.6%, Avg loss: 2.462918 

Epoch 2
-------------------------------
loss: 2.464949  [    0/60000]
loss: 2.359627  [ 6400/60000]
loss: 2.453544  [12800/60000]
loss: 2.574821  [19200/60000]
loss: 2.536742  [25600/60000]
loss: 2.359425  [32000/60000]
loss: 2.382489  [38400/60000]
loss: 2.519785  [44800/60000]
loss: 2.582898  [51200/60000]
loss: 2.435494  [57600/60000]
Test Error: 
 Accuracy: 8.4%, Avg loss: 2.462702 

Epoch 3
-------------------------------
loss: 2.471209  [    0/60000]
loss: 2.330177  [ 6400/60000]
loss: 2.450504  [12800/60000]
loss: 2.580200  [19200/60000]
loss: 2.469589  [25600/60000]
loss: 2.369143  [32000/60000

**Expected output**

In theory, you should see that the larger dropout rate you use, the lower test accuracy you will get, at the same epoch number.

But the model trained with some dropout rate should generalize better to new data.